# 第十章 · 在T4上做蒸馏

[![在Colab中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zxh0/vibe-writing/blob/main/toyllm_book2/code/book/ch10/colab.ipynb)

和前几章的笔记本一样，只做杂事：挂载Google Drive、拉代码，然后调用`book/ch10/`里的脚本。

**起点是第六章的基座模型**，要在Drive的`我的云端硬盘/toyllm2/runs/pretrain/model/`下。三个学生都从它出发。

**老师是Qwen3-0.6B**，第一次运行时从HuggingFace下载（约1.5GB），用Colab自带的`transformers`跑。

分四步：

1. **挑问题**：从smol-smoltalk里挑2万个训练用、500个验证用的问题，连同原来的回答，存到Drive的`toyllm2/data/distill/prompts.jsonl`。
2. **老师写回答**：估计一个小时上下。回答随时备份到Drive的`toyllm2/data/distill/teacher.jsonl`，断了重跑会接着写。
3. **装数据**：过滤老师的回答，装成三份：老师的回答、原来的回答、只取5千个问题的老师回答。
4. **训三个学生**，再比一比。

**用法**：菜单“代码执行程序 → 更改运行时类型”选T4 GPU，改好第一个格子里的配置，然后“全部运行”。
断线以后重新连上，再“全部运行”一次，做完的会跳过。

In [ ]:
# ---- 配置 ----------------------------------------------------------------
REPO_URL = "https://github.com/zxh0/vibe-writing.git"
BRANCH = "main"
SUBDIR = "toyllm_book2/code"
DRIVE_DIR = "/content/drive/MyDrive/toyllm2"

BASE_RUN = "pretrain"     # 学生的起点：runs/<BASE_RUN>/model
TEACHER = "Qwen/Qwen3-0.6B"
N_TRAIN = 20000           # 挑几个训练用的问题
TEACHER_ARGS = []         # 额外的参数，比如 ["--batch", "32"]
# 三个学生：名字、数据、额外的参数。数据少，过3遍；验证集只有一百多行。
SFT_ARGS = ["--epochs", 3, "--warmup", 30, "--eval-every", 30, "--eval-batches", 5,
            "--sample-every", 100, "--ckpt-every", 50, "--export"]
STUDENTS = [("distill", "teacher"), ("distill-orig", "orig"), ("distill-5k", "teacher-5k")]

In [ ]:
import os, pathlib, shutil, subprocess, sys

def sh(cmd, cwd=None):
    """跑一条shell命令，边跑边打印，失败就停。"""
    print("$", cmd, flush=True)
    stream(cmd, shell=True, cwd=cwd)

def stream(cmd, **kw):
    """子进程的输出一行一行转给print：Colab不一定显示子进程直接写到
    终端的输出，转一道才能保证在格子里看得到。"""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, **kw)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)

import torch
assert torch.cuda.is_available(), "没有CUDA：菜单“代码执行程序 → 更改运行时类型”选GPU"
p = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability()
print(f"GPU    {p.name}，sm{cap[0]}{cap[1]}，显存{p.total_memory / 2**30:.1f}GiB")
print(f"torch  {torch.__version__}，CUDA {torch.version.cuda}，Python {sys.version.split()[0]}")
print(f"CPU    {os.cpu_count()}个线程")
import numpy, pyarrow, safetensors, tokenizers, transformers   # 都是Colab自带的
print(f"transformers {transformers.__version__}")

## 挂载Drive，拉代码

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = pathlib.Path(DRIVE_DIR)
RUNS = DRIVE / "runs"
BASE = RUNS / BASE_RUN / "model"
assert (BASE / "model.safetensors").exists(), f"Drive上没有{BASE}，先跑第六章"
SAVED = DRIVE / "data" / "distill"
SAVED.mkdir(parents=True, exist_ok=True)

# 稀疏克隆：整个仓库有好几本书，这里只取第十章的代码。
WORK = pathlib.Path("/content/vibe-writing")
if not WORK.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} --filter=blob:none --no-checkout {REPO_URL} {WORK}")
    sh(f"git sparse-checkout set --no-cone /{SUBDIR}/book/ch10/", cwd=WORK)
    sh("git checkout", cwd=WORK)
else:
    sh("git pull --ff-only", cwd=WORK)
sh("git log -1 --format='%h %ad %s' --date=short", cwd=WORK)
CODE = WORK / SUBDIR
TOKENIZER = BASE / "tokenizer.json"

def run(*args):
    """在code/下跑book/ch10/里的一个脚本，输出边跑边打印。"""
    cmd = [sys.executable, "-u", f"book/ch10/{args[0]}", *map(str, args[1:])]
    print("$ python", " ".join(cmd[2:]), flush=True)
    stream(cmd, cwd=CODE)

## 挑问题

几十秒，外加下载smol-smoltalk（约1GB）。Drive上已经有`prompts.jsonl`就跳过：老师已经写了一部分时，问题不能变。

In [ ]:
if (SAVED / "prompts.jsonl").exists():
    print("prompts.jsonl已经有了，跳过")
else:
    run("pick_prompts.py", "--n", N_TRAIN, "--raw", "/content/data/smol-smoltalk",
        "--out", SAVED, "--tokenizer", TOKENIZER)

## 老师写回答

估计一个小时上下，每批打印一行进度和预计还要多久。回答写在本地盘，每10批整个复制一份到Drive的`toyllm2/data/distill/teacher.jsonl`。断了重跑这个格子，会先从Drive复制回来，已经写过的问题跳过。

In [ ]:
DATA = pathlib.Path("/content/data/distill")
DATA.mkdir(parents=True, exist_ok=True)
shutil.copy(SAVED / "prompts.jsonl", DATA / "prompts.jsonl")
run("teacher.py", "--model", TEACHER, "--prompts", DATA / "prompts.jsonl",
    "--out", DATA / "teacher.jsonl", "--backup", SAVED / "teacher.jsonl", *TEACHER_ARGS)

## 装数据

过滤老师的回答，装成三份，存在本地盘`/content/data/distill/`下，几十秒。

In [ ]:
run("prepare_distill.py", "--data", DATA, "--tokenizer", TOKENIZER)

## 训三个学生

每个几分钟。训完存成模型目录：`toyllm2/runs/<名字>/model/`，训完的跳过。

In [ ]:
for name, data in STUDENTS:
    if (RUNS / name / "model").exists():
        print(f"{name}已经训完，跳过")
        continue
    run("sft.py", "--name", name, "--init", BASE, "--data", DATA / data, "--runs", RUNS,
        "--resume", *SFT_ARGS)

## 比一比

交叉的验证loss，以及每个学生回答验证集前100个问题时的长度、自己停下的比例和重复。然后问几个问题。

In [ ]:
models = [RUNS / name / "model" for name, _ in STUDENTS]
run("compare.py", *models, "--data", DATA)
run("chat.py", "--model", *models)